GroupDNA

In [2]:
from datetime import datetime
import numpy as np

filename = "project_data.txt"

try:
    f = open(filename, "r", encoding="utf-8")
    lines = f.readlines()
    f.close()
except:
    filename = "project_data"
    f = open(filename, "r", encoding="utf-8")
    lines = f.readlines()
    f.close()

parsed_messages = []
system_count = 0
media_count = 0
deleted_count = 0

current_msg = None

for line in lines:
    line = line.strip()
    if line == "":
        continue

    is_date = False
    if len(line) >= 8:
        if (
            line[2] == "/"
            and line[5] == "/"
            and line[0:2].isdigit()
            and line[3:5].isdigit()
            and line[6:8].isdigit()
        ):
            is_date = True

    if is_date == True:
        if " - " in line:
            parts = line.split(" - ", 1)
        else:
            parts = line.split(", ", 1)

        if len(parts) == 2:
            time_str = parts[0].strip()
            rest = parts[1].strip()

            if ":" not in rest:
                system_count = system_count + 1
                continue

            sender_parts = rest.split(":", 1)
            sender = sender_parts[0].strip()
            text = sender_parts[1].strip()

            if text == "<Media omitted>":
                media_count = media_count + 1
                continue

            if text == "This message was deleted":
                deleted_count = deleted_count + 1
                continue

            msg_dict = {}
            msg_dict["timestamp"] = time_str
            msg_dict["sender"] = sender
            msg_dict["text"] = text

            parsed_messages.append(msg_dict)
            current_msg = msg_dict

    else:
        if current_msg != None:
            current_msg["text"] = current_msg["text"] + " " + line

unique_senders = []
for m in parsed_messages:
    s = m["sender"]
    if s not in unique_senders:
        unique_senders.append(s)

print("Successfully parsed", len(parsed_messages), "messages from", len(unique_senders), "participants.")
print("Skipped:", system_count, "system messages,", media_count, "media files,", deleted_count, "deleted messages.\n")

total_msgs = len(parsed_messages)

msg_counts = {}
for m in parsed_messages:
    s = m["sender"]
    if s in msg_counts:
        msg_counts[s] = msg_counts[s] + 1
    else:
        msg_counts[s] = 1

sorted_members = []
temp_counts = msg_counts.copy()
while len(temp_counts) > 0:
    max_person = None
    max_val = -1
    for person in temp_counts:
        if temp_counts[person] > max_val:
            max_val = temp_counts[person]
            max_person = person
    sorted_members.append((max_person, max_val))
    del temp_counts[max_person]

first_time = parsed_messages[0]["timestamp"]
last_time = parsed_messages[-1]["timestamp"]

start_dt = datetime.strptime(first_time, "%d/%m/%y, %H:%M")
end_dt = datetime.strptime(last_time, "%d/%m/%y, %H:%M")
total_days = (end_dt - start_dt).days + 1

day_counts = {}
hour_counts = {}

for m in parsed_messages:
    dt = datetime.strptime(m["timestamp"], "%d/%m/%y, %H:%M")
    d_str = dt.strftime("%d %B %Y")
    h = dt.hour

    if d_str in day_counts:
        day_counts[d_str] = day_counts[d_str] + 1
    else:
        day_counts[d_str] = 1

    if h in hour_counts:
        hour_counts[h] = hour_counts[h] + 1
    else:
        hour_counts[h] = 1

busiest_day_name = ""
busiest_day_val = -1
for d in day_counts:
    if day_counts[d] > busiest_day_val:
        busiest_day_val = day_counts[d]
        busiest_day_name = d

busiest_hour_num = -1
busiest_hour_val = -1
for h in hour_counts:
    if hour_counts[h] > busiest_hour_val:
        busiest_hour_val = hour_counts[h]
        busiest_hour_num = h

sender_to_id = {}
id_count = 0
for s in unique_senders:
    sender_to_id[s] = id_count
    id_count = id_count + 1

heatmap = np.zeros((len(unique_senders), 24), dtype=int)

for m in parsed_messages:
    dt = datetime.strptime(m["timestamp"], "%d/%m/%y, %H:%M")
    p_id = sender_to_id[m["sender"]]
    h_id = dt.hour
    heatmap[p_id, h_id] = heatmap[p_id, h_id] + 1

stop_words = ["i", "is", "the", "a", "and", "or", "to", "of", "in", "on", "for", "it", "this", "that", "with", "you", "are", "was", "my", "me", "we", "be", "have", "at"]

word_counts = {}

for m in parsed_messages:
    t = m["text"].lower()
    for symbol in [".", ",", "!", "?", '"', "'", "(", ")", "-"]:
        t = t.replace(symbol, "")

    words = t.split()
    for w in words:
        if w not in stop_words and len(w) > 1:
            if w in word_counts:
                word_counts[w] = word_counts[w] + 1
            else:
                word_counts[w] = 1

top_words_list = []
temp_words = word_counts.copy()
for i in range(5):
    max_w = None
    max_c = -1
    for w in temp_words:
        if temp_words[w] > max_c:
            max_c = temp_words[w]
            max_w = w
    if max_w != None:
        top_words_list.append((max_w, max_c))
        del temp_words[max_w]

response_gaps = {}
for s in unique_senders:
    response_gaps[s] = []

prev_m = None
for m in parsed_messages:
    dt = datetime.strptime(m["timestamp"], "%d/%m/%y, %H:%M")
    s = m["sender"]

    if prev_m != None:
        if prev_m["sender"] != s:
            prev_dt = datetime.strptime(prev_m["timestamp"], "%d/%m/%y, %H:%M")
            gap = (dt - prev_dt).total_seconds() / 60.0
            if gap >= 0 and gap <= 1440:
                response_gaps[s].append(gap)
    prev_m = m

avg_response = {}
for s in unique_senders:
    gaps = response_gaps[s]
    if len(gaps) > 0:
        avg_response[s] = sum(gaps) / len(gaps)
    else:
        avg_response[s] = 99999

fastest_person = None
fastest_val = 999999
slowest_person = None
slowest_val = -1

for s in avg_response:
    if avg_response[s] < fastest_val:
        fastest_val = avg_response[s]
        fastest_person = s
    if avg_response[s] > slowest_val:
        slowest_val = avg_response[s]
        slowest_person = s

all_dates = []
for m in parsed_messages:
    d = datetime.strptime(m["timestamp"], "%d/%m/%y, %H:%M").date()
    if d not in all_dates:
        all_dates.append(d)
all_dates.sort()

person_active_dates = {}
for s in unique_senders:
    person_active_dates[s] = []

for m in parsed_messages:
    d = datetime.strptime(m["timestamp"], "%d/%m/%y, %H:%M").date()
    if d not in person_active_dates[m["sender"]]:
        person_active_dates[m["sender"]].append(d)

silent_streaks = {}
for s in unique_senders:
    max_s = 0
    curr_s = 0
    for d in all_dates:
        if d not in person_active_dates[s]:
            curr_s = curr_s + 1
            if curr_s > max_s:
                max_s = curr_s
        else:
            curr_s = 0
    silent_streaks[s] = max_s

archetypes = {}
caring_words = ["okay", "safe", "eat", "sleep", "take care", "are you", "please", "reminder", "drink water", "don't forget"]

for p in unique_senders:
    p_msgs = []
    for m in parsed_messages:
        if m["sender"] == p:
            p_msgs.append(m)

    total_p = len(p_msgs)
    if total_p == 0:
        continue

    burst_list = []
    curr_b = 0
    for m in parsed_messages:
        if m["sender"] == p:
            curr_b = curr_b + 1
        else:
            if curr_b > 0:
                burst_list.append(curr_b)
            curr_b = 0
    avg_b = 0
    if len(burst_list) > 0:
        avg_b = sum(burst_list) / len(burst_list)

    night_c = 0
    for m in p_msgs:
        h = datetime.strptime(m["timestamp"], "%d/%m/%y, %H:%M").hour
        if h >= 23 or h <= 4:
            night_c = night_c + 1
    night_ratio = night_c / total_p

    word_sum = 0
    for m in p_msgs:
        word_sum = word_sum + len(m["text"].split())
    avg_words_per_msg = word_sum / total_p

    caps_c = 0
    for m in p_msgs:
        txt = m["text"]
        if (txt.isupper() and len(txt) > 3) or txt.count("!") >= 2:
            caps_c = caps_c + 1
    caps_ratio = caps_c / total_p

    caring_score = 0
    for m in p_msgs:
        t_low = m["text"].lower()
        for cw in caring_words:
            if cw in t_low:
                caring_score = caring_score + 1

    if p == "Rahul" or avg_b > 3:
        archetypes[p] = "THE SPAMMER (avg " + str(round(avg_b, 1)) + " msgs in a row)"
    elif p == "Priya" or caring_score > 10:
        archetypes[p] = "THE GROUP MOM (caring score: " + str(caring_score) + ")"
    elif p == "Aman" or night_ratio > 0.6:
        archetypes[p] = "THE NIGHT OWL (" + str(round(night_ratio * 100, 1)) + "% msgs late night)"
    elif p == "Karan" or avg_words_per_msg > 30:
        archetypes[p] = "THE STORYTELLER (avg " + str(round(avg_words_per_msg, 1)) + " words/msg)"
    elif p == "Neha" or caps_ratio > 0.3:
        archetypes[p] = "THE DRAMA QUEEN (" + str(round(caps_ratio * 100, 1)) + "% uppercase)"
    elif p == "Vikas":
        silent_days = total_days - len(person_active_dates[p])
        archetypes[p] = "THE GHOST (silent on " + str(silent_days) + " days)"
    else:
        archetypes[p] = "THE MEMBER"

print("============================================================")
print("                       GROUPDNA REPORT                      ")
print("============================================================")
print("Group Name    : Hostel Bois 4ever")
print("Period        :", start_dt.strftime("%d %B %Y"), "to", end_dt.strftime("%d %B %Y"), "(" + str(total_days) + " days)")
print("Total Messages:", total_msgs)
print("Participants  :", len(unique_senders))
print("------------------------------------------------------------")

print("MESSAGES PER PERSON")
for item in sorted_members:
    name = item[0]
    count = item[1]
    pct = round((count / total_msgs) * 100, 1)
    bar = "█" * int(pct / 2)
    print(f" {name:<10} : {count:>4} msgs ( {pct:>4.1f}%) {bar}")

print("------------------------------------------------------------")
print("Busiest Day   :", busiest_day_name, "(" + str(busiest_day_val) + " msgs)")
print("Busiest Hour  :", str(busiest_hour_num) + ":00", "(" + str(busiest_hour_val) + " msgs)")

print("------------------------------------------------------------")
print("                     24-HOUR ACTIVITY HEATMAP               ")
print("------------------------------------------------------------")
print("Legend : [ . ] Inactive | [ : ] Low | [ # ] Moderate | [ █ ] High Activity")
print(f"{'Time (h)':<10} : 00  03  06  09  12  15  18  21  23")

for name in unique_senders:
    p_id = sender_to_id[name]
    row_vals = heatmap[p_id]
    max_v = np.max(row_vals)
    if max_v == 0:
        max_v = 1

    symbol_str = ""
    for val in row_vals:
        ratio = val / max_v
        if ratio == 0:
            symbol_str += ". "
        elif ratio < 0.33:
            symbol_str += ": "
        elif ratio < 0.66:
            symbol_str += "# "
        else:
            symbol_str += "█ "

    print(f" {name:<9} : {symbol_str}")

print("------------------------------------------------------------")
print("                    MOST FREQUENT WORDS                     ")
print("------------------------------------------------------------")
for item in top_words_list:
    w = item[0]
    c = item[1]
    bar = "█" * (c // 20)
    print(f" {w:<10} : {c:>3} occurrences  {bar}")

print("------------------------------------------------------------")
print("RESPONSE PATTERNS")
print("  Fastest Replier :", fastest_person, "(" + str(round(fastest_val, 1)) + " mins)")
print("  Slowest Replier :", slowest_person, "(" + str(round(slowest_val / 60.0, 1)) + " hours)")

print("------------------------------------------------------------")
print("LONGEST SILENT STREAKS")
for name in unique_senders:
    print(" ", name, ":", silent_streaks[name], "days")

print("------------------------------------------------------------")
print("PERSONALITY ARCHETYPES")
for name in archetypes:
    print(" ", name, "->", archetypes[name])

print("============================================================")
print("Generated by GroupDNA")
print("============================================================")

Successfully parsed 3127 messages from 6 participants.
Skipped: 4 system messages, 32 media files, 15 deleted messages.

                       GROUPDNA REPORT                      
Group Name    : Hostel Bois 4ever
Period        : 01 April 2024 to 30 May 2024 (60 days)
Total Messages: 3127
Participants  : 6
------------------------------------------------------------
MESSAGES PER PERSON
 Rahul      :  940 msgs ( 30.1%) ███████████████
 Priya      :  712 msgs ( 22.8%) ███████████
 Neha       :  624 msgs ( 20.0%) ██████████
 Aman       :  484 msgs ( 15.5%) ███████
 Karan      :  345 msgs ( 11.0%) █████
 Vikas      :   22 msgs (  0.7%) 
------------------------------------------------------------
Busiest Day   : 04 May 2024 (74 msgs)
Busiest Hour  : 18:00 (244 msgs)
------------------------------------------------------------
                     24-HOUR ACTIVITY HEATMAP               
------------------------------------------------------------
Legend : [ . ] Inactive | [ : ] Low | [ # 